# Ontology in AI (Artificial Intelligence, the field of building computer systems that do tasks needing human-like understanding)

Welcome to a gentle, hands-on introduction to one of the most useful ideas from the research on ontologies and AI. An ontology is a shared, written-down vocabulary for a topic: it lists the important kinds of things, how they relate, and the rules they must follow, so that both people and computers mean the same thing when they use the same words. By the end of this notebook you will be able to explain what an ontology contains, how it differs from a simple hierarchy and from a database of facts, how facts can be written as triples, how two vocabularies can be matched and checked for mistakes, and how a statistical text generator can use an ontology without inventing nonsense.

A few shortenings we will use everywhere, each explained here once so nothing stays mysterious: an LLM (Large Language Model, a statistical text generator trained on very large collections of text) is the text generator we want to keep honest; OWL (Web Ontology Language, a formal vocabulary format for writing ontologies so computers can reason with them) is the standard format for writing ontologies; RDF (Resource Description Framework, a simple format for facts written as three-part triples) is the simple triple format underneath; SPARQL (SPARQL Protocol and RDF Query Language, a query language for asking questions over triple data) is the language for asking questions over those triples; a KG (Knowledge Graph, a database of entities and the relationships between them) is the database of facts built with the vocabulary; a CQ (Competency Question, a plain test question that an ontology must be able to answer) is the test question an ontology must answer; and I/O (Input/Output, reading data in and writing data out) below is only simulated with in-memory dictionaries so this notebook runs with the standard library alone and never touches the network.

This tutorial follows the condensed synthesis in [the topic digest](../digest.md), which itself summarizes eight paper summaries kept next to this research folder.

## What an ontology is, and what it is not

An ontology has four building blocks, and each one earns its place. Classes are the kinds of things in the topic, such as Pet or Cat, arranged from general to specific so that everything true of a Pet is also true of a Cat. Properties are the named relationships and attributes, such as chases or hasName, and each property declares which classes it connects, called its domain and its range. Individuals are the concrete members, such as a particular cat named Tom, each belonging to one or more classes. Axioms are the rules of the game, such as every Cat is a Pet, or nothing can be both a Cat and a Dog, and these rules let a computer spot contradictions and fill in facts that were never written down explicitly. The research synthesis stresses that these four pieces support the full engineering lifecycle from requirements through publication to maintenance, with the text generator acting as drafter, domain expert, and reviewer rather than as the final authority ([the topic digest](../digest.md); [systematic review](../../../LargeLanguageModelsForOntologyEngineering/summary.md)).

A taxonomy is only the skeleton of this richer structure: it gives the parent-to-child tree, such as Animal to Pet to Cat, but it says nothing about relationships like chases, nothing about concrete members, and nothing about rules, so it cannot answer whether a statement is contradictory or which questions the vocabulary should handle. A KG (Knowledge Graph, defined above) is the opposite complement: it is the large collection of concrete facts written with the vocabulary, such as Tom chases Luna, while the ontology is the vocabulary and the rules that keep those facts consistent. The benchmark literature shows why the distinction matters in practice: generators recover single terms easily yet collapse on relations, hierarchies, and full rules, so a vocabulary without axioms and a fact collection without a vocabulary both fail in different ways ([benchmark study](../../../CQ4OE/summary.md); [French legal graph study](../../../FrenchLegalKnowledgeGraph/summary.md); [the topic digest](../digest.md)).

In [ ]:
# Tiny hand-built ontology using only the standard library.
# Simulated I/O (Input/Output, defined above): load_ontology_slice pretends
# to read from storage but returns an in-memory dictionary, so nothing
# leaves this notebook and no network or secret keys are ever needed.

def load_ontology_slice():
    return {
        "classes": {"Animal": None, "Pet": "Animal", "Cat": "Pet", "Dog": "Pet"},
        "properties": {
            "chases": {"domain": "Pet", "range": "Pet"},
            "hasName": {"domain": "Animal", "range": "Text"},
        },
        "individuals": [
            {"name": "tom", "class": "Cat"},
            {"name": "luna", "class": "Cat"},
        ],
        "axioms": ["Every Cat is a Pet", "Nothing is both a Cat and a Dog"],
    }

ontology = load_ontology_slice()
print("classes:", ontology["classes"])
print("properties:", list(ontology["properties"]))
print("individuals:", [i["name"] for i in ontology["individuals"]])
print("axioms:", ontology["axioms"])

## Facts as triples: the RDF and OWL idea

Underneath the friendly dictionaries above sits a beautifully simple idea that the whole field shares. Every fact is written as a triple with three parts: a subject, a predicate, and an object, such as Tom, chases, Luna. The subject names the thing we talk about, the predicate names the relationship from our fixed vocabulary, and the object names the other thing or value, which means every sentence in our database has exactly the same shape and a computer can scan millions of them without parsing grammar. OWL (Web Ontology Language, defined above) adds the vocabulary discipline on top of these triples: it declares which predicates are relationships between members versus attributes carrying plain values, which classes sit under which parents, and which class each relationship may start from and end at, so that a question asked in ordinary words can be translated into an exact query over the triples. The query research shows that placing this complete typed vocabulary directly into the generator context makes single-call translation from ordinary questions into exact queries reliable, while informal schema dumps and raw file pastes fail, especially on domain and range discipline ([query execution study](../../../NaturalLanguageKnowledgeGraphQueryExecution/summary.md); [metadata access study](../../../NaturalLanguageAccessToDomainSpecificMetadata/summary.md); [the topic digest](../digest.md)).

In [ ]:
triples = [(s, "type", "Cat") for s in ["tom", "luna"]]
triples += [("tom", "chases", "luna"), ("Cat", "subClassOf", "Pet")]
triples += [(p, "domain", "Pet") for p in ["chases"]]
for t in triples: print(t[0], "-", t[1], "->", t[2])
print(len(triples), "triples total")

## Matching vocabularies and catching mistakes

Real projects rarely enjoy a single clean vocabulary, because different teams invent different words for the same things and the same word for different things, so two skills keep every project alive. Alignment is the craft of matching one vocabulary against another, deciding for instance that Kitty in the second vocabulary means Cat in the first, and the tooling research shows that disciplined one-to-one matching with stable pairing lifts match quality dramatically while blind guessing and raw file pastes fail systematically ([alignment study](../../../OpenOntologies/summary.md); [the topic digest](../digest.md)). Consistency checking is the guard at the gate: it replays every new fact against the declared domains, ranges, hierarchies, and rules, flagging anything that breaks them, such as a relationship used between classes it was never declared for, a member claimed under two disjoint classes, or a newly invented predicate nobody approved. Production pipelines therefore wrap generation in deterministic normalization and deduplication plus symbolic gates and human review, which in one reported deployment cut invented facts to zero on a hostile test collection while lifting search recall substantially ([extraction layer study](../../../OntologyGuidedDeduplicationAwareExtractionLayer/summary.md); [extension framework study](../../../OntoExtend/summary.md); [French legal graph study](../../../FrenchLegalKnowledgeGraph/summary.md); [the topic digest](../digest.md)).

In [ ]:
# Toy alignment plus consistency checking, standard library only.
# Simulated I/O: both vocabularies are plain dictionaries standing in
# for records that would normally be loaded from storage.
vocab_a = {"Cat": "small feline pet", "chases": "runs after"}
vocab_b = {"Kitty": "small feline pet", "pursues": "runs after"}
meaning_to_a = {}
for label, meaning in vocab_a.items():
    meaning_to_a.setdefault(meaning, label)
alignment, taken = {}, set()
for label_b, meaning_b in vocab_b.items():
    label_a = meaning_to_a.get(meaning_b)
    if label_a is not None and label_a not in taken:
        alignment[label_a] = label_b
        taken.add(label_a)
print("alignment:", alignment)
hierarchy = {"Cat": "Pet", "Pet": "Animal", "Furniture": None}
member_of = {"tom": "Cat", "luna": "Cat", "stool": "Furniture"}
domains = {"chases": "Pet"}
def is_kind(child, parent):
    while child is not None:
        if child == parent:
            return True
        child = hierarchy.get(child)
    return False
facts = [("tom", "chases", "luna"), ("stool", "chases", "tom")]
bad = [f for f in facts if not is_kind(member_of[f[0]], domains[f[1]])]
print("violations:", bad)
assert alignment == {"Cat": "Kitty", "chases": "pursues"}
assert bad == [("stool", "chases", "tom")]
print("alignment and consistency demonstration passed")

## How a text generator uses an ontology without inventing facts

A text generator left to itself is a fluent drafter and a poor librarian: it proposes terms quickly but stumbles on relations, hierarchies, and domain and range discipline, so unconstrained output can never be merged directly. Three patterns from the research tame it, and they all share one idea of grounding every step in the controlled vocabulary. First, constrained generation, sometimes called constrained decoding, only lets the generator pick classes, properties, and members from a retrieved slice of the ontology, reusing existing elements instead of inventing new ones, which keeps answers inside the approved vocabulary even when the full ontology is far larger than the context window ([extension framework study](../../../OntoExtend/summary.md); [extraction layer study](../../../OntologyGuidedDeduplicationAwareExtractionLayer/summary.md); [the topic digest](../digest.md)). Second, KG grounding (Knowledge Graph grounding, answering from retrieved facts rather than from memory) fetches the relevant subgraph before answering, so that questions are answered from stored triples instead of from the haze of training memory, with deterministic rewriting bridging friendly names to native predicates when the underlying store uses opaque codes ([query execution study](../../../NaturalLanguageKnowledgeGraphQueryExecution/summary.md); [metadata access study](../../../NaturalLanguageAccessToDomainSpecificMetadata/summary.md)). Third, honest evaluation asks test questions the ontology must answer, compares generated structures against references with term and axiom scores, and routes every change through symbolic gates and human ratings before merging, because the literature finds thin human evidence and nonstandard metrics wherever this discipline is skipped ([benchmark study](../../../CQ4OE/summary.md); [systematic review](../../../LargeLanguageModelsForOntologyEngineering/summary.md); [the topic digest](../digest.md)).

In [ ]:
# Failure case: one ambiguous word, two meanings, and the fix.
# Simulated I/O: the retrieved slice stands in for records fetched
# from storage; everything runs offline with the standard library.
text = "The workers press the grapes every harvest"
unconstrained = {"press": "publishes"}  # wrong sense, guessed from newspaper habits
print("unconstrained mapping:", unconstrained)
allowed = {"classes": ["Device", "Grape"], "properties": ["crushes"]}
context_words = set(text.lower().split())
if "grapes" in context_words or "harvest" in context_words:
    constrained = {"press": "crushes"}  # wine-press device sense, inside the slice
else:
    constrained = {"press": "publishes"}
print("ontology slice:", allowed)
print("constrained mapping:", constrained)
assert constrained["press"] in allowed["properties"]
assert unconstrained["press"] not in allowed["properties"]
print("fix confirmed: grounding in the slice picks the harvest sense")

## What to remember, and where to read next

Carry home four sentences and the whole field fits in your pocket. An ontology is the agreed vocabulary plus the rules, with classes for kinds, properties for relationships, individuals for concrete members, and axioms for the rules that let computers check consistency. A hierarchy alone cannot express relationships or rules, and a fact collection without a vocabulary cannot stay consistent, which is why the ontology and the KG (Knowledge Graph, defined above) need each other. Triples give every fact the same three-part shape, the typed vocabulary turns ordinary questions into exact queries, and matching plus checking keep merged vocabularies honest. A text generator becomes trustworthy only when it must choose from the retrieved vocabulary slice, answer from stored facts, and pass gates and human review before anything is merged.

When you are ready for depth, start with [the topic digest](../digest.md), then follow the leaves: the lifecycle survey in [the systematic review](../../../LargeLanguageModelsForOntologyEngineering/summary.md), the single-call query mechanism in [the query execution study](../../../NaturalLanguageKnowledgeGraphQueryExecution/summary.md), the vocabulary-versus-structure collapse in [the benchmark study](../../../CQ4OE/summary.md), the readable-vocabulary findings in [the metadata access study](../../../NaturalLanguageAccessToDomainSpecificMetadata/summary.md), the matching results in [the alignment study](../../../OpenOntologies/summary.md), the scaling pipeline in [the French legal graph study](../../../FrenchLegalKnowledgeGraph/summary.md), the per-question extension method in [the extension framework study](../../../OntoExtend/summary.md), and the production guardrails in [the extraction layer study](../../../OntologyGuidedDeduplicationAwareExtractionLayer/summary.md). Finish by running the quiz cell below: if every check passes, you have earned the concepts.

In [ ]:
# Closing quiz: each line states one thing you should now know.
assert ontology["classes"]["Cat"] == "Pet", "Cat must sit under Pet"
assert ontology["properties"]["chases"]["domain"] == "Pet", "chases starts from Pet"
assert any(i["name"] == "tom" for i in ontology["individuals"]), "Tom must be a member"
assert len(triples) == 5, "the triple demonstration holds five triples"
assert alignment.get("Cat") == "Kitty", "alignment must match Cat to Kitty"
assert bad == [("stool", "chases", "tom")], "the checker must catch the stool fact"
assert constrained["press"] == "crushes", "grounding must pick the harvest sense"
print("All quiz checks passed: classes, properties, members, triples,")
print("alignment, consistency, and grounded generation are understood.")